# Task 1: Inconsistencies and duplicates
**Name:** Fares Mohamed Shahin  
**ID:** 61-1799  
**Course:** CSEN1095 Data Engineering, W26

In [1]:
import re
import pandas as pd

raw = pd.read_csv('club_signups.csv')   # the original file is only read, never changed
df = raw.copy()                          # every fix below is done on this copy

## Part A: Inconsistencies
### A1. Look first

In [2]:
print('shape:', raw.shape)
print()
print(raw.dtypes)

shape: (39, 9)

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object


In [3]:
for col in ['faculty', 'club', 'city', 'fee_paid']:
    print(raw[col].value_counts(dropna=False))
    print()

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club    1
Music          1
Debate         1
Name: count, dtype: int64

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO          1

The problems are visible: the same faculty, club and city are written in several ways (case, extra spaces, long names, nicknames), `fee_paid` has many spellings, and `signed_up_at` is still text.

### A2. Faculty, club and city
First a mechanical clean (strip spaces, lower-case). Then an explicit dictionary for what is left.

In [4]:
def mechanical(s):
    # strip, collapse repeated spaces, lower-case
    return s.str.strip().str.replace(r'\s+', ' ', regex=True).str.lower()

faculty_map = {
    'met': 'MET', 'media engineering and technology': 'MET',
    'iet': 'IET', 'information engineering and technology': 'IET',
    'ems': 'EMS', 'engineering and materials science': 'EMS',
    'pharmacy': 'Pharmacy', 'pharma': 'Pharmacy',
    'management': 'Management',
}
club_map = {
    'robotics': 'Robotics',
    'debate': 'Debate', 'debating': 'Debate', 'debate club': 'Debate',
    'music': 'Music',
    'football': 'Football', 'soccer': 'Football',   # Soccer club = Football club
    'chess': 'Chess', 'chess club': 'Chess',
}
city_map = {
    'cairo': 'Cairo',
    'giza': 'Giza', 'el giza': 'Giza',
    'alexandria': 'Alexandria', 'alex': 'Alexandria',
}

df['faculty'] = mechanical(df['faculty']).map(faculty_map)
df['club']    = mechanical(df['club']).map(club_map)
df['city']    = mechanical(df['city']).map(city_map)

# a value missing from a dictionary becomes NaN, so check that nothing was missed
assert df[['faculty', 'club', 'city']].notna().all().all(), 'some value is not in the dictionaries'
assert set(df['faculty']) <= {'MET', 'IET', 'EMS', 'Pharmacy', 'Management'}
assert set(df['club'])    <= {'Robotics', 'Debate', 'Music', 'Football', 'Chess'}
assert set(df['city'])    <= {'Cairo', 'Giza', 'Alexandria'}

for col in ['faculty', 'club', 'city']:
    print(df[col].value_counts()); print()

faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64

club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64

city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64



### A3. Names and emails
Names: remove extra spaces and use the same capitalisation for everyone (Title Case). Emails: trimmed and lower-case.

In [5]:
df['full_name'] = df['full_name'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
df['email'] = df['email'].str.strip().str.lower()

assert not df['full_name'].str.contains(r'^\s|\s$|\s\s').any()
assert (df['email'] == df['email'].str.strip().str.lower()).all()
print(df[['student_id', 'full_name', 'email']].drop_duplicates().sort_values('student_id').to_string(index=False))

student_id      full_name                             email
   55-2639  Youssef Fathy  youssef.fathy@student.guc.edu.eg
   55-2971  Salma Ibrahim  salma.ibrahim@student.guc.edu.eg
   55-2992   Mohamed Adel   mohamed.adel@student.guc.edu.eg
   55-9245    Ali Ibrahim    ali.ibrahim@student.guc.edu.eg
   58-1089    Karim Kamel    karim.kamel@student.guc.edu.eg
   58-1125     Ziad Saleh     ziad.saleh@student.guc.edu.eg
   58-8701 Farida Ibrahim farida.ibrahim@student.guc.edu.eg
   58-9118   Mariam Kamel   mariam.kamel@student.guc.edu.eg
   61-2495     Ahmed Adel     ahmed.adel@student.guc.edu.eg
   61-3846     Ahmed Zaki     ahmed.zaki@student.guc.edu.eg
   61-4733      Omar Adel      omar.adel@student.guc.edu.eg
   61-4844   Mohamed Adel   mohamed.adel@student.guc.edu.eg
   61-7344  Laila Mostafa  laila.mostafa@student.guc.edu.eg
   61-8224      Sara Zaki      sara.zaki@student.guc.edu.eg
   61-8942      Hana Zaki      hana.zaki@student.guc.edu.eg


### A4. `fee_paid` to a boolean

In [6]:
print(raw['fee_paid'].astype(str).str.strip().str.lower().unique())

fee_map = {'yes': True, 'y': True, 'true': True, '1': True,
           'no': False, 'n': False, 'false': False, '0': False}
df['fee_paid'] = df['fee_paid'].astype(str).str.strip().str.lower().map(fee_map)

assert df['fee_paid'].notna().all()
df['fee_paid'] = df['fee_paid'].astype(bool)
print(df['fee_paid'].dtype)
print(df['fee_paid'].value_counts())

['n' 'false' 'y' 'yes' '1' 'true' '0' 'no']
bool
fee_paid
True     27
False    12
Name: count, dtype: int64


### A5. `signed_up_at`: two formats
The column mixes `2026-09-15 13:39` (year first) and `15/09/2026 03:12` (with slashes). For the slash format I need to decide which part is the day and which is the month. I check the first number.

In [7]:
s = raw['signed_up_at'].str.strip()
is_iso = s.str.match(r'^\d{4}-\d{2}-\d{2} ')       # 2026-09-15 13:39
is_slash = s.str.match(r'^\d{2}/\d{2}/\d{4} ')     # 15/09/2026 03:12
print('year-first rows:', is_iso.sum(), '| slash rows:', is_slash.sum(), '| unmatched:', (~is_iso & ~is_slash).sum())

first = s[is_slash].str.slice(0, 2).astype(int)
second = s[is_slash].str.slice(3, 5).astype(int)
print('first number in slash dates:', sorted(first.unique()))
print('second number in slash dates:', sorted(second.unique()))

year-first rows: 25 | slash rows: 14 | unmatched: 0
first number in slash dates: [np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]
second number in slash dates: [np.int64(9)]


The first number goes up to 18 (a month cannot be above 12) while the second number is always 09. So the slash format is **day/month/year**, and the year-first format is year-month-day. All dates fall between 13 and 18 September 2026, which agrees with this.

In [8]:
parsed_iso = pd.to_datetime(s[is_iso], format='%Y-%m-%d %H:%M')
parsed_slash = pd.to_datetime(s[is_slash], format='%d/%m/%Y %H:%M')
df['signed_up_at'] = pd.concat([parsed_iso, parsed_slash]).sort_index()

assert df['signed_up_at'].notna().all()
print(df['signed_up_at'].dtype)
print(df['signed_up_at'].min(), '->', df['signed_up_at'].max())

datetime64[ns]
2026-09-13 14:36:00 -> 2026-09-18 23:36:00


## Part B: Duplicates
### B1. Exact duplicates
Same form submission sent twice: every column is identical (same `signup_id` too).

In [9]:
n_start = len(df)
n_exact = df.duplicated().sum()
print('rows at the start:', n_start)
print('exact duplicates:', n_exact)
print(df[df.duplicated(keep=False)].sort_values('signup_id').to_string(index=False))

df = df.drop_duplicates().reset_index(drop=True)
n_after_exact = len(df)
print('rows left:', n_after_exact)

rows at the start: 39
exact duplicates: 3
signup_id student_id  full_name                         email  faculty  city     club        signed_up_at  fee_paid
     S015    61-8942  Hana Zaki  hana.zaki@student.guc.edu.eg Pharmacy  Giza Football 2026-09-15 13:39:00     False
     S015    61-8942  Hana Zaki  hana.zaki@student.guc.edu.eg Pharmacy  Giza Football 2026-09-15 13:39:00     False
     S022    61-8224  Sara Zaki  sara.zaki@student.guc.edu.eg      EMS  Giza    Music 2026-09-16 01:39:00      True
     S022    61-8224  Sara Zaki  sara.zaki@student.guc.edu.eg      EMS  Giza    Music 2026-09-16 01:39:00      True
     S023    58-1125 Ziad Saleh ziad.saleh@student.guc.edu.eg      IET Cairo    Chess 2026-09-16 09:29:00     False
     S023    58-1125 Ziad Saleh ziad.saleh@student.guc.edu.eg      IET Cairo    Chess 2026-09-16 09:29:00     False
rows left: 36


### B2. Same sign-up submitted again
Some students signed up for the same club twice with a different `signup_id` (usually to mark the fee as paid). Keep the **latest** submission for each `student_id` + `club`.

In [10]:
again = df[df.duplicated(['student_id', 'club'], keep=False)].sort_values(['student_id', 'club', 'signed_up_at'])
print(again[['signup_id', 'student_id', 'full_name', 'club', 'signed_up_at', 'fee_paid']].to_string(index=False))

signup_id student_id     full_name     club        signed_up_at  fee_paid
     S020    55-9245   Ali Ibrahim    Music 2026-09-15 23:07:00     False
     S033    55-9245   Ali Ibrahim    Music 2026-09-16 09:07:00      True
     S021    61-4844  Mohamed Adel   Debate 2026-09-16 01:15:00     False
     S035    61-4844  Mohamed Adel   Debate 2026-09-17 11:15:00      True
     S017    61-7344 Laila Mostafa   Debate 2026-09-15 18:01:00     False
     S034    61-7344 Laila Mostafa   Debate 2026-09-16 02:01:00      True
     S032    61-7344 Laila Mostafa Football 2026-09-17 23:36:00     False
     S036    61-7344 Laila Mostafa Football 2026-09-18 23:36:00      True


In [11]:
# make sure no two submissions of the same student and club have exactly the same time
assert not df.duplicated(['student_id', 'club', 'signed_up_at']).any()

df = df.sort_values('signed_up_at')
df = df.drop_duplicates(['student_id', 'club'], keep='last')   # keep the latest
df = df.sort_values('signup_id').reset_index(drop=True)

n_resubmitted = n_after_exact - len(df)
print('removed as re-submissions:', n_resubmitted)
print('rows left:', len(df))

removed as re-submissions: 4
rows left: 32


### B3. Check

In [12]:
assert not df.duplicated(['student_id', 'club']).any(), 'student_id + club is not unique'
print('student_id + club is unique')

student_id + club is unique


## Order of steps: what if the duplicates were removed before fixing the club spellings?
For the README I compare with the wrong order, using the raw (uncleaned) `club` values.

In [13]:
wrong = raw.drop_duplicates()                                        # exact duplicates on raw data
print('exact duplicates found on raw data :', len(raw) - len(wrong))
wrong_after = wrong.drop_duplicates(['student_id', 'club'], keep='last')  # raw club text, order of file
print('same-club re-submissions found on raw club spellings:', len(wrong) - len(wrong_after))
print('rows left with the wrong order:', len(wrong_after), '| rows left with the right order:', len(df))

exact duplicates found on raw data : 3
same-club re-submissions found on raw club spellings: 2
rows left with the wrong order: 34 | rows left with the right order: 32


## Two different students with the same name
Mohamed Adel appears with two different student ids. I use `student_id` (not the name) as the key.

In [14]:
same_name = df[df['full_name'] == 'Mohamed Adel']
print(same_name[['signup_id', 'student_id', 'faculty', 'city', 'club']].to_string(index=False))
print()
print('rows with student_id + club as key:', len(df))
print('rows if the name was used instead :', len(df.drop_duplicates(['full_name', 'club'], keep='last')))

signup_id student_id faculty       city     club
     S002    55-2992     IET Alexandria   Debate
     S008    61-4844     EMS      Cairo    Music
     S009    61-4844     EMS      Cairo Robotics
     S024    55-2992     IET Alexandria Robotics
     S035    61-4844     EMS      Cairo   Debate

rows with student_id + club as key: 32
rows if the name was used instead : 30


## Save the result

In [15]:
df = df[['signup_id', 'student_id', 'full_name', 'email', 'faculty', 'city', 'club', 'signed_up_at', 'fee_paid']]
df.to_csv('club_signups_clean.csv', index=False)

check = pd.read_csv('club_signups_clean.csv')
print(check.shape)
check.head(10)

(32, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
5,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
9,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False
